In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Linear (or Conv) → Batch Normalization → Activation

# Instance of Modules

In [3]:
# -----------------------------------------------------------------------------------------------
class linear:
  
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5 # note: kaiming init
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])

# -----------------------------------------------------------------------------------------------
class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      if x.ndim == 2:
        dim = 0
      elif x.ndim == 3:
        dim = (0,1)
      xmean = x.mean(dim, keepdim=True) # batch mean
      xvar = x.var(dim, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

# -----------------------------------------------------------------------------------------------
class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

# -----------------------------------------------------------------------------------------------
class Embedding:
  
  def __init__(self, num_embeddings, embedding_dim):
    self.weight = torch.randn((num_embeddings, embedding_dim))
    
  def __call__(self, IX):
    self.out = self.weight[IX]
    return self.out
  
  def parameters(self):
    return [self.weight]

# -----------------------------------------------------------------------------------------------
class FlattenConsecutive:
  
  def __init__(self, n):
    self.n = n
    
  def __call__(self, x):
    B, T, C = x.shape
    x = x.view(B, T//self.n, C*self.n)
    if x.shape[1] == 1:
      x = x.squeeze(1)
    self.out = x
    return self.out
  
  def parameters(self):
    return []

# -----------------------------------------------------------------------------------------------
class Sequential:
  
  def __init__(self, layers):
    self.layers = layers
  
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    self.out = x
    return self.out
  
  def parameters(self):
    # get parameters of all layers and stretch them out into one list
    return [p for layer in self.layers for p in layer.parameters()]

## torch.nn.Linear from scratch 
<a href="https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html">Documentation</a> <br>
class torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None)

In [33]:
# intialsize init
g = torch.Generator().manual_seed(42)
m = nn.Linear(10,20,bias=False)
W1 = m.weight
x = torch.randn((4,10),generator = g)
y = m(x)

* class torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None)[source]
Applies an affine linear transformation to the incoming data: 
 $ y = xA^T + b $

In [46]:
W1.shape # the weight matrxi is of shape [out,in];

torch.Size([20, 10])

In [47]:
g = torch.Generator().manual_seed(42)
W = W1.T
x = torch.randn((4,10),generator = g)
h = x @ W 
(h == y).all()

tensor(True)

## torch.nn.BatchNorm1d from scratch
* (without running mean and variance)
<a href="https://docs.pytorch.org/docs/2.12/generated/torch.nn.BatchNorm1d.html">Documentation</a> <br>
class torch.nn.BatchNorm1d(num_features, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, device=None, dtype=None, *, bias=True)

### 🧠 Why do we normalize down the columns (across the batch) instead of along the rows?

If you normalize along the **rows**, you are actually doing **Layer Normalization**! 

While normalizing along rows is a completely valid and highly popular technique (used extensively in models like Transformers), it serves a different purpose than **Batch Normalization**. Here is exactly why Batch Norm goes down the columns (across the batch).

---

### 🍎 The Real-World Analogy: Mixing Apples and Oranges

To understand why we don't look across rows in Batch Norm, let's look at what the rows and columns actually represent. Imagine your batch of data contains health metrics for **4 different patients (rows)**, and you have **3 features (columns)**: *Age*, *Annual Income ($)*, and *Heart Rate (BPM)*.

| Patient (Rows) | Feature 1: Age | Feature 2: Income | Feature 3: Heart Rate |
| :--- | :---: | :---: | :---: |
| **Patient 1** | 25 | 60,000 | 72 |
| **Patient 2** | 45 | 120,000 | 65 |
| **Patient 3** | 19 | 15,000 | 80 |
| **Patient 4** | 60 | 85,000 | 68 |

#### ❌ What happens if you normalize along the ROW? (Layer Norm)
If you calculate the mean and variance for **Patient 1** along their row, you are mathematically mashing together completely different units:
$$\text{25 (years)} + \text{60,000 (dollars)} + \text{72 (beats per minute)}$$

Because the income ($\$60,000$) is massively larger than age or heart rate, it completely dominates the calculation. Normalizing this row squashes the variation of the other features into irrelevance. *(Note: Layer Norm works beautifully inside deep hidden layers because activation values usually share similar scales, but for raw or distinct features, it can be problematic).*

####  What happens if you normalize down the COLUMN? (Batch Norm)
If you look down the **Age** column, you are comparing numbers that all speak the same language:
$$25, 45, 19, 60 \text{ (years)}$$

By calculating the mean and variance of *just this column*, you can scale the values so that a patient's age is easily comparable to the rest of the population in that batch. You then do the exact same thing independently for the Income column and the Heart Rate column.

---

### 📊 Summary of the Dimensional Difference

[Image of Batch Normalization vs Layer Normalization dimensions]

* **Batch Normalization (`dim=0`):** For every single feature, look at all samples in the batch. Make sure that *this specific feature* behaves consistently across the data.
* **Layer Normalization (`dim=1`):** For a single sample, look at all of its features at once. Make sure the *overall scale* of activations for this single example doesn't blow up.

In [51]:
## Redefining linear layer
g = torch.Generator().manual_seed(42)
m = nn.Linear(10,20,bias=False)
W = m.weight
x = torch.randn((4,10),generator = g)
x = m(x) # preactivation and prebatch norm values 
## batchnorm
# number of features = 20
bn = nn.BatchNorm1d(20,momentum=0.01)
hpreact = bn(x) # outputs a normalised layer just before activation .

In [67]:
print(hpreact.shape) # prediction (4,20)
# check
# mean = 0, var = 1 along row
print(hpreact.mean(0,keepdim=True)[0,0])

torch.Size([4, 20])
tensor(0., grad_fn=<SelectBackward0>)


In [127]:
g = torch.Generator().manual_seed(42)
x = torch.randn((4,10),generator = g)
x = x @ W.T

In [129]:
x.shape

torch.Size([4, 20])

In [130]:
print(x.mean(0,keepdims=True))
print(x.var(0))

tensor([[ 0.1697,  0.0850, -0.1986,  0.0386,  0.2835,  0.3179,  0.3190,  0.1463,
         -0.1798,  0.3526, -0.0948,  0.2839, -0.0095,  0.1023, -0.5888, -0.1786,
          0.1577,  0.3011, -0.0823,  0.3706]], grad_fn=<MeanBackward1>)
tensor([0.6676, 0.0954, 1.2450, 0.1246, 1.3324, 0.0983, 0.3925, 0.5058, 0.6796,
        0.4822, 0.8101, 0.6670, 0.1971, 0.1501, 0.5296, 0.5158, 0.5699, 0.0998,
        0.4128, 0.1986], grad_fn=<VarBackward0>)


In [112]:
x.shape

torch.Size([4, 20])

In [131]:
x = (x-x.mean(0,keepdim=True))/x.var(0,keepdim=True) #keepdim=True is healthiear

In [135]:
x.sum(0).sum()

tensor(6.3330e-08, grad_fn=<SumBackward0>)

In [136]:
gamma = torch.ones((1,20))
beta = torch.zeros((1,20))
h = gamma*x + beta

## Folding from scratch

In [22]:
class fuseblock(nn.Module):
    # Linear -> Batchnorm -> Tanh
    def __init__(self,fin,fout):
        super().__init__()
        self.linear = nn.Linear(fin,fout,bias=False)
        self.bn = nn.BatchNorm1d(fout)
        #activation
        self.act = nn.Tanh()
        self.is_fused = False
    def forward(self,x):
        if self.is_fused:
            return self.act(self.linear(x))
        else :
            return self.act(self.bn(self.linear(x)))
    def fuse(self):
        if self.is_fused:
            return
        self.eval()
        # params
        W = self.linear.weight.data
        b = self.linear.bias.data if self.linear.bias is not None else torch.zeros(W.shape[0])
        gamma = self.bn.weight
        beta = self.bn.bias
        rmean = self.bn.running_mean
        rvar = self.bn.running_var
        eps = self.bn.eps
        # scaling
        c = gamma/torch.sqrt(rvar+eps)
        self.linear.weight.data = W * c.unsqueeze(1)
        if self.linear.bias is None:
                self.linear.bias = nn.Parameter(torch.zeros(W.shape[0]))
        self.linear.bias.data = (b - rmean)*c + beta
        self.bn = nn.Identity()
        self.is_fused = True
        print("block fused batchnorm removed")

In [17]:
class FoldableMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = fuseblock(10, 20)
        ##self.block2 = FusableLinearBlock(20, 20)
        #self.classifier = nn.Linear(20, 2)
    def forward(self,x):
        x = self.block1(x)
        #x = self.block2(x)
        #return self.classifier(x)
        return x
    def fuse_net(self):
        self.block1.fuse()
        #self.block2.fuse()

In [26]:
## again
g = torch.Generator().manual_seed(42)
m = nn.Linear(10,20,bias=False)
W = m.weight
x = torch.randn((4,10),generator = g)
x = m(x) # preactivation and prebatch norm values 
## batchnorm
# number of features = 20
bn = nn.BatchNorm1d(20)

h = torch.tanh(bn(x))
h.data
# outputs a normalised layer just before activation .

tensor([[ 0.8597, -0.9370, -0.9295,  0.9189, -0.8821, -0.9223, -0.7109, -0.9293,
         -0.8110, -0.9294,  0.9333, -0.8821,  0.9318, -0.9118, -0.3195, -0.7530,
          0.7359, -0.9186, -0.8693,  0.9190],
        [ 0.4065,  0.3687,  0.0719, -0.6994,  0.7589, -0.0785,  0.9045,  0.5956,
          0.9194,  0.7035, -0.7321, -0.3881, -0.1884,  0.6270, -0.9006,  0.8068,
          0.7304,  0.8028, -0.5333, -0.2922],
        [-0.8928,  0.4845,  0.6384,  0.1361, -0.4755,  0.7101,  0.3118,  0.0782,
          0.0034,  0.6056, -0.2803,  0.4692, -0.7307, -0.2165,  0.8023,  0.7037,
         -0.4104,  0.4809,  0.7901, -0.8276],
        [-0.2803,  0.6626,  0.6791, -0.6926,  0.7206,  0.6615, -0.7307,  0.7107,
         -0.4290,  0.0778, -0.4315,  0.8580, -0.5011,  0.7705,  0.6053, -0.7668,
         -0.8927, -0.0508,  0.6928, -0.1008]])

In [30]:
model = FoldableMLP()
model.eval()
g = torch.Generator().manual_seed(42)
x = torch.randn((4,10),generator = g)
with torch.no_grad():
    out_before = model(x)
model.fuse_net()
with torch.no_grad():
    out_after = model(x)
    is_identical = torch.allclose(out_before, out_after, atol=1e-5)

print("\n--- Test Results ---")
print(f"Output Before Fusion:\n{out_before}\n")
print(f"Output After Fusion:\n{out_after}\n")
print(f"Are the outputs mathematically identical? {'✅ YES' if is_identical else '❌ NO'}")
model.block1.is_fused

block fused batchnorm removed

--- Test Results ---
Output Before Fusion:
tensor([[ 0.7346, -0.8803, -0.8584, -0.7477, -0.1718, -0.6433,  0.7614,  0.3207,
         -0.8184, -0.7005, -0.7686, -0.0628, -0.9600, -0.0126, -0.0428,  0.5886,
         -0.7315, -0.8858, -0.6187, -0.1178],
        [ 0.3801, -0.5327,  0.3262,  0.4006, -0.0306, -0.3921,  0.0779,  0.2921,
          0.2087,  0.2512, -0.0676,  0.3042,  0.4811, -0.5493,  0.1207,  0.2127,
          0.1735, -0.5150,  0.0058,  0.4579],
        [ 0.5516,  0.6564,  0.6427, -0.0494,  0.4586, -0.1445,  0.6940,  0.0408,
         -0.7522,  0.8368,  0.3613, -0.1885, -0.2639,  0.2831, -0.4567, -0.6787,
          0.6788,  0.7162,  0.7701,  0.4977],
        [-0.5409,  0.1725,  0.2016,  0.4523, -0.1802,  0.6958,  0.4385, -0.4638,
          0.0223,  0.4459,  0.6776,  0.7227,  0.8111, -0.6320,  0.7500,  0.1625,
          0.9267,  0.2499,  0.2945,  0.4610]])

Output After Fusion:
tensor([[ 0.7346, -0.8803, -0.8584, -0.7477, -0.1718, -0.6433,  0.7614,

True

## Keypoints

* unfused + train() ->safe training mode
* fused + train() -> unsafe
* check status of fused by calling model.block1.is_fused

* after training then fuse the network
* then model.eval()